# FakeDet VLM — backend nhanh trên Colab T4 (free)

Chạy model đã train trên **GPU T4 free** cho demo/bảo vệ. Output là 1 URL công khai → dán vào `fakedet_system/.env`.

Backend này phục vụ **CẢ 2 version** vision classifier:
- `fakedet-vlm` → v1 (best_model.pth, 98.8% acc)
- `fakedet-vlm-v2` → v2 (best_model_v2.pth, anti-overfit recipe)

open-webui sẽ hiện **3 model** trong dropdown:
- `fakedet-v1.2` / `fakedet-v1.2-colab` → vision v1
- `fakedet-v1.2-v2-colab` → vision v2

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU** → Save.

Tốc độ: ~2–3s/ảnh (vs ~30–60s trên HF Space CPU). Link sống ~12h. Bảo vệ xong → Runtime → Disconnect.

In [ ]:
# @title 1. Cài deps + cài fakedet_vlm từ GitHub
!pip install -q "transformers>=4.46,<4.50" "peft>=0.12,<0.14" bitsandbytes \
    "accelerate>=0.34,<1.0" "timm>=1.0.9" pillow "huggingface_hub>=0.24,<0.27" \
    sentencepiece protobuf "safetensors>=0.4" "fastapi>=0.115" "uvicorn>=0.30" \
    pydantic python-multipart pyngrok
!pip install -q --no-deps \
    "git+https://github.com/duongvan17/FakeImageDetector_v1.git#subdirectory=fakedet_vlm"
print('✅ deps installed')


In [ ]:
# @title 2. Đăng nhập HuggingFace (paste Read token)
# Token: https://huggingface.co/settings/tokens (quyền Read là đủ).
# Các repo model đang private nên BẮT BUỘC login.
from huggingface_hub import login
login()

In [ ]:
# @title 3. Tải checkpoints (~2.5GB, 2–3 phút)
from huggingface_hub import hf_hub_download, snapshot_download

# --- Vision v1 (dùng cho VLM vision tower + classifier v1) ---
hf_hub_download('duongvan17/fakedet-vit-b16-fakeclue', 'best_model.pth',
                local_dir='/content/clip_model')
print('✅ Vision v1 checkpoint sẵn sàng')

# --- Vision v2 (classifier v2 — anti-overfit recipe) ---
hf_hub_download('duongvan17/fakedet-vit-b16-fakeclue-v2', 'best_model.pth',
                local_dir='/content/clip_model_v2')
print('✅ Vision v2 checkpoint sẵn sàng')

# --- VLM adapter (stage2 LoRA + projector) ---
snapshot_download('duongvan17/fakedet-vlm-stage2', local_dir='/content/adapter',
                  allow_patterns=['adapter_*', 'projector.pt', 'tokenizer*',
                                  'special_*', 'added_*', 'vocab.json', 'merges.txt'])
print('✅ VLM adapter sẵn sàng')

In [ ]:
# @title 4. Khởi động backend + tunnel
# Chọn tunnel:
#   ngrok       -> URL CỐ ĐỊNH nếu có NGROK_DOMAIN (reserve sẵn ở dashboard).
#                  Cần Colab Secret `NGROK_AUTHTOKEN` (sidebar trái → 🔑 Secrets).
#   cloudflared -> URL RANDOM mỗi lần, không cần đăng ký gì.
TUNNEL          = "ngrok"  # @param ["ngrok", "cloudflared"]
NGROK_DOMAIN    = "nonsequent-frederic-nonceremonious.ngrok-free.dev"  # @param {type:"string"}
FAKE_POSITIVE   = "1"      # @param ["1", "0"]
MAX_NEW_TOKENS  = 96       # @param {type:"integer"}

import os, subprocess, time, re, threading

# --- Model paths ---
os.environ['FAKEDET_VISION_CHECKPOINT'] = '/content/clip_model/best_model.pth'
os.environ['FAKEDET_CLASSIFIER_CHECKPOINT_V2'] = '/content/clip_model_v2/best_model.pth'
os.environ['FAKEDET_ADAPTER_DIR']      = '/content/adapter'
os.environ['FAKEDET_PROJECTOR']        = '/content/adapter/projector.pt'
os.environ['FAKEDET_MAX_NEW_TOKENS']   = str(MAX_NEW_TOKENS)
os.environ['FAKEDET_LAZY_LOAD']        = '1'   # mở port ngay; nạp model lúc gọi đầu
os.environ['FAKEDET_FAKE_POSITIVE']    = FAKE_POSITIVE

# uvicorn nền
def _start_uvicorn():
    subprocess.run(['uvicorn', 'fakedet_vlm.serve.openai_api:app',
                    '--host', '0.0.0.0', '--port', '8000'])
threading.Thread(target=_start_uvicorn, daemon=True).start()
print('⏳ uvicorn đang khởi động (mở port ~20s)...')
time.sleep(20)

public_url = None

if TUNNEL == "ngrok":
    # Đọc authtoken từ Colab Secret (an toàn, không lưu vào notebook).
    try:
        from google.colab import userdata
        token = userdata.get('NGROK_AUTHTOKEN')
    except Exception:
        token = os.environ.get('NGROK_AUTHTOKEN', '')
    if not token:
        raise RuntimeError(
            "Thiếu NGROK_AUTHTOKEN. Vào sidebar Colab (🔑 Secrets) → "
            "Add new secret → Name=NGROK_AUTHTOKEN, Value=<authtoken>, "
            "bật toggle 'Notebook access'. Lấy authtoken ở: "
            "https://dashboard.ngrok.com/get-started/your-authtoken"
        )
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = token
    # Bind tới domain cố định nếu đã reserve ở dashboard ngrok; ko thì random.
    kw = {"proto": "http", "addr": 8000}
    if NGROK_DOMAIN.strip():
        kw["domain"] = NGROK_DOMAIN.strip()
    tun = ngrok.connect(**kw)
    public_url = tun.public_url

else:  # cloudflared (random URL, không cần đăng ký)
    if not os.path.exists('/content/cloudflared'):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
        !chmod +x /content/cloudflared
    proc = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True,
    )
    for _ in range(200):
        line = proc.stdout.readline()
        if not line: break
        print(line, end='')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0); break
    threading.Thread(
        target=lambda: [None for _ in iter(proc.stdout.readline, '')],
        daemon=True,
    ).start()

bar = '=' * 64
if public_url:
    fixed_note = ("(URL CỐ ĐỊNH — .env đã dán sẵn rồi)"
                  if TUNNEL == "ngrok" and NGROK_DOMAIN.strip()
                  else "(URL RANDOM — phải dán lại .env mỗi lần)")
    print(f"\n\n{bar}\n🌐  PUBLIC URL : {public_url}  {fixed_note}\n"
          f"📋  .env line:\n"
          f"      FAKEDET_BACKEND_URL_COLAB={public_url}/v1\n"
          f"\n"
          f"📋  Backend phục vụ 2 vision version:\n"
          f"      fakedet-vlm    → v1 classifier\n"
          f"      fakedet-vlm-v2 → v2 classifier (anti-overfit)\n"
          f"\n"
          f"📋  open-webui dropdown sẽ có 3 model:\n"
          f"      fakedet-v1.2          → HF Space, vision v1\n"
          f"      fakedet-v1.2-colab    → Colab, vision v1\n"
          f"      fakedet-v1.2-v2-colab → Colab, vision v2\n{bar}")
else:
    print('⚠️ Chưa tìm thấy URL — cuộn log phía trên.')


## Sau khi có URL

1. **Thêm / cập nhật** dòng trong `fakedet_system/.env`:
   ```
   FAKEDET_BACKEND_URL_COLAB=https://xxxx.ngrok-free.dev/v1
   ```
2. Restart container hệ thống:
   ```powershell
   cd fakedet_system
   docker compose restart fakedet-system
   ```
3. Mở <http://localhost:8088>. Dropdown model sẽ có **3 lựa chọn**:
   - `fakedet-v1.2` → HF Space, vision v1
   - `fakedet-v1.2-colab` → Colab T4, vision v1 (nhanh)
   - `fakedet-v1.2-v2-colab` → Colab T4, vision v2 (nhanh)

   Đổi backend/version chỉ bằng cách đổi model trong dropdown — không phải restart Docker.

## So sánh v1 vs v2

| | Vision v1 | Vision v2 |
|---|---|---|
| Checkpoint | `best_model.pth` | `best_model.pth` (từ repo v2) |
| Training | 1 LR group, basic aug | Diff LR, JPEG aug, RandAugment, label smooth |
| Precision/Recall | Precision 0.97, Recall 1.0 | Better balanced (anti-overfit) |
| Cross-domain | In-domain only | Mean(in-domain, OOD) |

## Khi xong

- Colab: Runtime → **Disconnect and delete runtime** (giải phóng quota GPU free).
- Sau đó model `-colab` sẽ ngưng phản hồi; chọn lại model không có `-colab` (HF) là xong.

In [ ]:
# @title 5. (Tuỳ chọn) Giữ session sống — chạy cell này rồi để yên
import time
while True:
    time.sleep(60)